# Project 2 — LendingClub Credit Funnel
## Notebook 02: Preprocessing, Feature Engineering & the scikit-learn Pipeline

**Author:** Eric Bowers · **Framework:** CRISP-DM (Data Preparation stage)

**BLUF.** This notebook turns the leakage-free feature whitelist from Notebook 01
into a **single, reproducible scikit-learn `Pipeline`**. Every feature-engineering
step — credit-history length, the `mths_since_*` "never" flags, log transforms,
rare-category grouping, missing-block flags — is implemented as a **custom
transformer that lives inside the pipeline and is fit on training data only**, so
there is no leakage from validation/test folds. The notebook then wires in a
baseline model, sets up **stratified** and **time-based** splits, handles class
imbalance, checks multicollinearity, and **serializes the fitted pipeline**
(pickled via `joblib`) for reproducible reuse in Notebook 03.

**Inputs (from Notebook 01):**
- `data/interim/accepted_terminal_whitelist.parquet` — terminal loans, whitelist features + `target`.
- `data/interim/column_audit.csv` — the leakage/keep disposition per column.

**Outputs:**
- `models/preprocess_baseline_pipeline.joblib` — the fitted FE + preprocessing + baseline pipeline.
- `src/lc_transformers.py` — the reusable custom transformers (imported here and in Notebook 03; **required on the import path to reload the pickle**).

## 0 · Setup

In [1]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
import joblib

sys.path.insert(0, "src")   # make the custom-transformer module importable (see §3)

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_val_score, RandomizedSearchCV)
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 200)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
import sklearn; print("scikit-learn", sklearn.__version__)

scikit-learn 1.6.1


### 0.1 · Configuration

`INCLUDE_LC_RISK` is the switch for the pitch's central experiment: the **primary
(independent) model excludes** LendingClub's own risk outputs (`grade`,
`sub_grade`, `int_rate`) so it can audit rather than echo them; the **benchmark
variant** sets this to `True` to quantify how much signal they add. Notebook 03
runs both and reports the ROC-AUC / PR-AUC gap.

In [2]:
INTERIM_DIR = r"C:\LendingClub\data\interim"
MODEL_DIR   = r"C:\LendingClub\models"
os.makedirs(MODEL_DIR, exist_ok=True)

PARQUET = os.path.join(INTERIM_DIR, "accepted_terminal_whitelist.parquet")
CSV_FALLBACK = PARQUET.replace(".parquet", ".csv")

INCLUDE_LC_RISK = False        # False = independent primary model (default)
LC_RISK_COLS = ["grade", "sub_grade", "int_rate", "int_rate_num"]

TEST_SIZE = 0.20
TIME_SPLIT_TRAIN_MAX_YEAR = 2016   # train <= 2016, test 2017-2018 (time-based split)

## 1 · Load the interim modeling frame

We load the whitelist frame from Notebook 01, split off the target and the
`issue_dt` timestamp (used *only* for the time-based split, never as a feature),
and — for the primary model — drop LendingClub's risk outputs.

In [3]:
if os.path.exists(PARQUET):
    df = pd.read_parquet(PARQUET)
elif os.path.exists(CSV_FALLBACK):
    df = pd.read_csv(CSV_FALLBACK)
else:
    raise FileNotFoundError("Run Notebook 01 first to create the interim file.")

print("Loaded:", df.shape)

# Target and the split-only timestamp.
y = df["target"].astype("int8")
issue_dt = pd.to_datetime(df["issue_dt"]) if "issue_dt" in df.columns else pd.NaT

# Features = everything except the target and the split-only timestamp.
drop_from_X = ["target", "issue_dt"]
X = df.drop(columns=[c for c in drop_from_X if c in df.columns])

# Primary model: drop LendingClub's own risk outputs (judgment-call features).
if not INCLUDE_LC_RISK:
    dropped = [c for c in LC_RISK_COLS if c in X.columns]
    X = X.drop(columns=dropped)
    print("Primary model — dropped LC risk outputs:", dropped)
else:
    print("Benchmark model — LC risk outputs retained.")

print("Feature matrix:", X.shape, "| default rate:", round(y.mean(), 4))

Loaded: (1348099, 91)
Primary model — dropped LC risk outputs: ['grade', 'sub_grade', 'int_rate', 'int_rate_num']
Feature matrix: (1348099, 85) | default rate: 0.1998


## 2 · Feature-engineering plan

Each step below is implemented as a transformer method so it runs inside the
pipeline and is **fit on training data only**:

| Step | Columns | Logic |
|---|---|---|
| Credit-history length | `earliest_cr_line`, `issue_d` | months between first credit line and issue → `credit_history_months`; raw dates dropped |
| "Ever had event" flags | `mths_since_last_delinq`, `mths_since_last_record`, `mths_since_last_major_derog` | `NaN = never` → binary `_ever` flag + numeric capped at `mths_cap` (NaN→cap = "never/long ago") |
| Employment length | `emp_length` | `"10+ years"…"< 1 year"` → ordinal 0–10 |
| Log transforms | monetary/skewed fields | `log1p` (values clipped at 0) → `_log`, raw dropped |
| Missing-block flags | ~38%-missing installment/revolving block | add `_missing` flag; keep numeric for imputation |
| Rare-category grouping | any high-cardinality categorical | keep top-N levels (learned on train), rest → `"Other"` |

The **numeric** branch then imputes (median) + scales; the **categorical** branch
groups rare levels + imputes (constant) + one-hot encodes.

In [4]:
# Column groups the feature engineer acts on are defined once in the module
# (imported here) so Notebooks 02 and 03 stay in sync. Each is applied only if
# the column is present, so unknown names are harmless no-ops.
from lc_transformers import (LOG_COLS, MTHS_SINCE_COLS,
                             INSTALLMENT_BLOCK, EMP_LENGTH_MAP)
print("LOG_COLS:", LOG_COLS)
print("MTHS_SINCE_COLS:", MTHS_SINCE_COLS)
print("INSTALLMENT_BLOCK:", INSTALLMENT_BLOCK)
print("EMP_LENGTH_MAP keys:", list(EMP_LENGTH_MAP))

LOG_COLS: ['annual_inc', 'loan_amnt', 'revol_bal', 'tot_cur_bal', 'total_bal_il', 'total_bal_ex_mort', 'tot_hi_cred_lim', 'avg_cur_bal', 'bc_open_to_buy']
MTHS_SINCE_COLS: ['mths_since_last_delinq', 'mths_since_last_record', 'mths_since_last_major_derog']
INSTALLMENT_BLOCK: ['open_il_12m', 'open_il_24m', 'il_util', 'all_util', 'total_bal_il', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc', 'open_acc_6m', 'inq_last_12m', 'total_cu_tl', 'inq_fi', 'mths_since_rcnt_il']
EMP_LENGTH_MAP keys: ['< 1 year', '1 year', '2 years', '3 years', '4 years', '5 years', '6 years', '7 years', '8 years', '9 years', '10+ years']


## 3 · Custom transformers (imported from `src/lc_transformers.py`)

The feature-engineering logic lives in a **module**, not in a notebook cell, and
we import it here. This is deliberate and matters for the assignment's
serialization requirement: `joblib` pickles a custom class **by its import path**.
A class defined in a notebook cell pickles as `__main__.CreditFeatureEngineer`
and **fails to reload** in Notebook 03 or a deployment script; a class in
`lc_transformers` pickles as `lc_transformers.CreditFeatureEngineer` and loads
anywhere the module is importable.

Three transformers, all fit on training data only (no target leakage, no
test→train statistics):
- **`CreditFeatureEngineer`** — credit-history length, `mths_since_*` "ever" flags
  + capped numeric, `emp_length` ordinal, log transforms, missing-block flags.
- **`RareCategoryGrouper`** — collapse rare categorical levels (top-N learned on
  train) to `"Other"`; unseen levels at transform time also map to `"Other"`.
- **`RiskMissingFlag`** — the `credit_missing` indicator for the policy model (§10).

The source is displayed below so the notebook is self-documenting.

In [5]:
from lc_transformers import (CreditFeatureEngineer, RareCategoryGrouper,
                             RiskMissingFlag)
print("Imported custom transformers from src/lc_transformers.py")

Imported custom transformers from src/lc_transformers.py


In [6]:
# Show the transformer source inline (self-documenting; edit the module to change).
import inspect
print(inspect.getsource(CreditFeatureEngineer))

class CreditFeatureEngineer(BaseEstimator, TransformerMixin):
    """Application-time feature engineering for LendingClub loans.

    All operations are guarded by column presence, so the transformer is robust
    to the exact whitelist produced upstream. Nothing here learns from the target
    and no statistic crosses from test to train. Returns a pandas DataFrame.

    Parameters
    ----------
    mths_cap : int
        Cap (and NaN sentinel) for the ``mths_since_*`` fields, where NaN means
        the event never occurred and is treated as "a very long time ago".
    log_cols : list[str]
        Skewed monetary columns to ``log1p``-transform (raw column dropped).
    mths_cols : list[str]
        ``mths_since_*`` columns to convert to an ``_ever`` flag + capped numeric.
    installment_block : list[str]
        ~38%-missing installment/revolving columns to add ``_missing`` flags for.
    emp_length_map : dict
        Mapping from ``emp_length`` labels to an ordinal 0-10.
    """

 

In [7]:
print(inspect.getsource(RareCategoryGrouper))

class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    """Keep the top-N levels per categorical column (learned on train); others -> 'Other'.

    Unseen levels at transform time also map to the ``other_label``.
    """

    def __init__(self, top_n=15, other_label="Other"):
        self.top_n = top_n
        self.other_label = other_label

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.columns_ = list(X.columns)
        self.top_levels_ = {}
        for c in self.columns_:
            vc = X[c].astype("object").value_counts()
            self.top_levels_[c] = set(vc.head(self.top_n).index)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for c in self.columns_:
            keep = self.top_levels_.get(c, set())
            X[c] = X[c].astype("object").where(X[c].isin(keep), self.other_label)
        return X

    def get_feature_names_out(self, input_features=None):
        return np.array(input_features if inp

## 4 · Assemble the preprocessing `ColumnTransformer`

After feature engineering, columns are selected **by dtype** (`make_column_selector`)
so the pipeline adapts automatically to whatever engineered columns exist:

- **Numeric** → median impute → standard scale.
- **Categorical** → rare-group → constant-impute (`"missing"`) → one-hot (`handle_unknown="ignore"`).

`remainder="drop"` guarantees nothing un-typed (e.g. a stray datetime) reaches the
model.

In [8]:
numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("rare", RareCategoryGrouper(top_n=15)),
    ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=True, min_frequency=20)),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, make_column_selector(dtype_include=np.number)),
        ("cat", categorical_pipe, make_column_selector(dtype_include=["object", "category"])),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

# Feature engineering + preprocessing as one reusable block.
fe_prep = Pipeline([
    ("fe", CreditFeatureEngineer(
        mths_cap=180, log_cols=LOG_COLS, mths_cols=MTHS_SINCE_COLS,
        installment_block=INSTALLMENT_BLOCK, emp_length_map=EMP_LENGTH_MAP)),
    ("prep", clone(preprocess)),
])
fe_prep

Pipeline(steps=[('fe',
                 CreditFeatureEngineer(emp_length_map={'1 year': 1,
                                                       '10+ years': 10,
                                                       '2 years': 2,
                                                       '3 years': 3,
                                                       '4 years': 4,
                                                       '5 years': 5,
                                                       '6 years': 6,
                                                       '7 years': 7,
                                                       '8 years': 8,
                                                       '9 years': 9,
                                                       '< 1 year': 0},
                                       installment_block=['open_il_12m',
                                                          'open_il_24m',
                                                          'il_util', 'all_util',
                                                          'total_bal_il',
                                                          'open_rv_12m',
                                                          'open_rv_24m',
                                                          'max_bal_bc',
                                                          'open_acc_6m',
                                                          'inq_last_12m',
                                                          'total_cu_tl'...
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x0000014FCBBAB340>),
                                                 ('cat',
                                                  Pipeline(steps=[('rare',
                                                                   RareCategoryGrouper()),
                                                                  ('impute',
                                                                   SimpleImputer(fill_value='missing',
                                                                                 strategy='constant')),
                                                                  ('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 min_frequency=20))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x0000014FD0400850>)]))])

## 5 · Splits — stratified holdout and time-based split

Two evaluation regimes, per the pitch:
1. **Stratified holdout** preserves the ~20% default rate (primary reporting).
2. **Time-based split** trains on 2007–2016 vintages and tests on 2017–2018 —
   the realistic deployment scenario, and a drift probe. *Read the time-based
   test cautiously: recent 60-month vintages are censored by the terminal-loan
   filter (see Notebook 01, §2.1), so its test set skews toward faster-resolving
   loans.*

In [9]:
# 1) Stratified holdout.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
print(f"Stratified — train {X_train.shape}, test {X_test.shape} | "
      f"train default {y_train.mean():.3f}, test default {y_test.mean():.3f}")

# 2) Time-based split (only if we have usable timestamps).
def time_based_split(X, y, issue_dt, train_max_year):
    yr = pd.to_datetime(issue_dt).dt.year
    tr = yr <= train_max_year
    te = yr > train_max_year
    return X[tr], X[te], y[tr], y[te], (tr.sum(), te.sum())

if not pd.isna(issue_dt).all():
    Xtr_t, Xte_t, ytr_t, yte_t, sizes = time_based_split(
        X, y, issue_dt, TIME_SPLIT_TRAIN_MAX_YEAR)
    print(f"Time-based — train<= {TIME_SPLIT_TRAIN_MAX_YEAR}: {sizes[0]:,} rows "
          f"({ytr_t.mean():.3f} default) | test: {sizes[1]:,} rows ({yte_t.mean():.3f} default)")

Stratified — train (1078479, 85), test (269620, 85) | train default 0.200, test default 0.200
Time-based — train<= 2016: 1,122,460 rows (0.197 default) | test: 225,639 rows (0.213 default)


## 6 · Model integration + baseline

Integrating the estimator into the same pipeline means preprocessing is re-fit
inside every CV fold — the correct, leak-free pattern. The baseline is
**Logistic Regression with `class_weight="balanced"`** (the simplest imbalance
handling; alternatives compared in §7).

In [10]:
baseline = Pipeline([
    ("fe", CreditFeatureEngineer(
        mths_cap=180, log_cols=LOG_COLS, mths_cols=MTHS_SINCE_COLS,
        installment_block=INSTALLMENT_BLOCK, emp_length_map=EMP_LENGTH_MAP)),
    ("prep", clone(preprocess)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced",
                               solver="saga", n_jobs=-1, random_state=RANDOM_STATE)),
])

baseline.fit(X_train, y_train)
proba = baseline.predict_proba(X_test)[:, 1]
print(f"Baseline LogReg — ROC-AUC {roc_auc_score(y_test, proba):.4f} | "
      f"PR-AUC {average_precision_score(y_test, proba):.4f}")

Baseline LogReg — ROC-AUC 0.7148 | PR-AUC 0.3800


In [11]:
# How many features did preprocessing produce?
feat_names = baseline.named_steps["prep"].get_feature_names_out()
print(f"Engineered + encoded feature count: {len(feat_names)}")
print("Sample:", list(feat_names[:10]))

Engineered + encoded feature count: 138
Sample: ['num__installment', 'num__dti', 'num__delinq_2yrs', 'num__fico_range_low', 'num__fico_range_high', 'num__inq_last_6mths', 'num__mths_since_last_delinq', 'num__mths_since_last_record', 'num__open_acc', 'num__pub_rec']


In [12]:
# Cross-validated ROC-AUC (all preprocessing re-fit inside each fold).
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_auc = cross_val_score(baseline, X_train, y_train, cv=cv,
                         scoring="roc_auc", n_jobs=-1)
print(f"5-fold CV ROC-AUC: {cv_auc.mean():.4f} +/- {cv_auc.std():.4f}")

5-fold CV ROC-AUC: 0.7157 +/- 0.0007


## 7 · Class-imbalance handling — two routes

**Route A (default): `class_weight="balanced"`** — used above; nothing leaves the
pipeline, works cleanly inside CV.

**Route B: SMOTE resampling** via `imbalanced-learn`'s pipeline, which only
resamples the **training** portion of each fold (never validation/test). Use
`imblearn.pipeline.Pipeline` so the `SMOTE` step is skipped at predict time.
We compare the two empirically and keep whichever gives better PR-AUC in
Notebook 03.

In [13]:
try:
    from imblearn.pipeline import Pipeline as ImbPipeline
    from imblearn.over_sampling import SMOTE

    smote_pipe = ImbPipeline([
        ("fe", CreditFeatureEngineer(
            mths_cap=180, log_cols=LOG_COLS, mths_cols=MTHS_SINCE_COLS,
            installment_block=INSTALLMENT_BLOCK, emp_length_map=EMP_LENGTH_MAP)),
        ("prep", clone(preprocess)),
        ("smote", SMOTE(random_state=RANDOM_STATE)),          # train-fold only
        ("clf", LogisticRegression(max_iter=1000, solver="saga",
                                   n_jobs=-1, random_state=RANDOM_STATE)),
    ])
    smote_pipe.fit(X_train, y_train)
    p = smote_pipe.predict_proba(X_test)[:, 1]
    print(f"SMOTE + LogReg — ROC-AUC {roc_auc_score(y_test, p):.4f} | "
          f"PR-AUC {average_precision_score(y_test, p):.4f}")
except ImportError:
    print("imbalanced-learn not installed — `pip install imbalanced-learn` to run Route B.")

c:\Users\Orion\anaconda3\envs\ai-environment\lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


SMOTE + LogReg — ROC-AUC 0.7117 | PR-AUC 0.3787


## 8 · Multicollinearity check

The pitch calls for a multicollinearity check. We compute VIF on the transformed
numeric block (falling back to a high-correlation scan if `statsmodels` is
unavailable). High-VIF features are candidates for pruning before the interpretable
Logistic Regression, though tree/boosting models are largely unaffected.

In [14]:
# Transform a sample through FE+preprocess, then inspect numeric collinearity.
vif_pipe = clone(fe_prep)   # independent copy -> never touches `baseline`
Xt = vif_pipe.fit_transform(X_train.sample(min(20000, len(X_train)), random_state=RANDOM_STATE))
names = vif_pipe.named_steps["prep"].get_feature_names_out()
Xt_df = pd.DataFrame(Xt.toarray() if hasattr(Xt, "toarray") else Xt, columns=names)
num_cols = [c for c in Xt_df.columns if c.startswith("num__")]

try:
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    sub = Xt_df[num_cols].dropna()
    vif = pd.Series(
        [variance_inflation_factor(sub.values, i) for i in range(sub.shape[1])],
        index=num_cols).sort_values(ascending=False)
    print("Highest VIFs (>10 suggests redundancy):")
    print(vif.head(12))
except Exception as e:
    print(f"statsmodels unavailable ({e}); using correlation scan instead.")
    corr = Xt_df[num_cols].corr().abs()
    high = (corr.where(~np.eye(len(corr), dtype=bool))
                .stack().sort_values(ascending=False))
    print("Top correlated numeric pairs:")
    print(high.head(10))

c:\Users\Orion\anaconda3\envs\ai-environment\lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)
c:\Users\Orion\anaconda3\envs\ai-environment\lib\site-packages\statsmodels\regression\linear_model.py:1784: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.uncentered_tss


Highest VIFs (>10 suggests redundancy):
num__open_rv_12m_missing     inf
num__revol_util_num          inf
num__open_il_12m_missing     inf
num__open_il_24m_missing     inf
num__open_rv_24m_missing     inf
num__max_bal_bc_missing      inf
num__open_acc_6m_missing     inf
num__inq_last_12m_missing    inf
num__revol_util              inf
num__fico_mid                inf
num__total_cu_tl_missing     inf
num__fico_range_high         inf
dtype: float64


## 9 · Serialize the pipeline (pickled, reproducible)

We persist the **entire fitted pipeline** — feature engineering, preprocessing,
and the baseline estimator — with `joblib`. Because every step is inside the
pipeline, loading this one file reproduces the exact transform + prediction on new
application-time data, with no separate preprocessing code to keep in sync.

> **To reload elsewhere:** the loading process must be able to `import
> lc_transformers` (i.e. run `sys.path.insert(0, "src")` first), because the
> pickle references the custom classes by that import path. This is exactly why
> §3 imports them from a module rather than defining them in a cell.

In [15]:
model_path = os.path.join(MODEL_DIR, "preprocess_baseline_pipeline.joblib")
# Guard: prep output width must match the classifier before we pickle.
_n_out = baseline[:-1].transform(X_train.head(1000)).shape[1]
assert _n_out == baseline.named_steps["clf"].n_features_in_, (
    f"prep emits {_n_out} cols but clf expects {baseline.named_steps['clf'].n_features_in_}"
    " -- a shared step was refit; restart kernel and Run All.")
joblib.dump(baseline, model_path)
print("Saved:", model_path,
      f"({os.path.getsize(model_path)/1024**2:.1f} MB)")

# Also record which columns the pipeline expects, for the model card / README.
schema = {
    "n_features_in": int(X.shape[1]),
    "include_lc_risk": bool(INCLUDE_LC_RISK),
    "input_columns": list(X.columns),
    "sklearn_version": sklearn.__version__,
}
with open(os.path.join(MODEL_DIR, "pipeline_input_schema.json"), "w") as f:
    json.dump(schema, f, indent=2)
print("Saved input schema.")

Saved: C:\LendingClub\models\preprocess_baseline_pipeline.joblib (0.0 MB)
Saved input schema.


In [16]:
# Reproducibility check: reload and confirm identical predictions.
reloaded = joblib.load(model_path)
p1 = baseline.predict_proba(X_test.head(1000))[:, 1]
p2 = reloaded.predict_proba(X_test.head(1000))[:, 1]
assert np.allclose(p1, p2), "Reloaded pipeline predictions differ!"
print("Reload check passed — predictions identical after serialization.")

Reload check passed — predictions identical after serialization.


## 10 · Reuse for the policy-characterization model (secondary)

The *same* preprocessing pattern applies to the access/policy model on the
harmonized accepted-vs-rejected columns (Notebook 01, §10): numeric impute+scale,
categorical group+impute+one-hot, with a `Risk_Score_missing` flag. The helper
below builds that lighter preprocessing block so Notebook 03's policy study reuses
the identical, leak-free machinery. Recall the policy model **describes historical
approval**, it does not predict future approvals or infer declined-applicant
outcomes.

In [17]:
def build_policy_preprocess(add_risk_missing_flag=True):
    """ColumnTransformer for the harmonized approved-vs-rejected policy model.

    Uses the module-level RiskMissingFlag so a fitted policy pipeline is also
    pickle-safe across notebooks.
    """
    return Pipeline([
        ("flag", RiskMissingFlag(add_flag=add_risk_missing_flag, credit_col="credit")),
        ("prep", ColumnTransformer([
            ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                              ("scale", StandardScaler())]),
             make_column_selector(dtype_include=np.number)),
            ("cat", Pipeline([("rare", RareCategoryGrouper(top_n=15)),
                              ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
                              ("ohe", OneHotEncoder(handle_unknown="ignore"))]),
             make_column_selector(dtype_include=["object", "category"])),
        ], remainder="drop")),
    ])

print("Policy preprocessing builder ready (used in Notebook 03).")

Policy preprocessing builder ready (used in Notebook 03).


## 11 · Summary → Notebook 03 (Modeling)

**Built here**
- `CreditFeatureEngineer` + `RareCategoryGrouper` (in `src/lc_transformers.py`) —
  all feature engineering inside the pipeline, fit on train only (no leakage),
  and importable so the pickled pipeline reloads across notebooks.
- A dtype-driven `ColumnTransformer` (median-impute + scale / rare-group +
  constant-impute + one-hot) that adapts to the engineered columns.
- Stratified **and** time-based splits; a baseline LogReg integrated end-to-end;
  CV ROC-AUC computed with preprocessing re-fit per fold.
- Imbalance routes (class weight vs. SMOTE), a multicollinearity check, and a
  **serialized, reload-verified** pipeline.

**Next (Notebook 03)**
- Swap the estimator for **Random Forest** and **Gradient Boosting (XGBoost/LightGBM)**
  in the same pipeline; run randomized → focused hyperparameter search on
  ROC-AUC/PR-AUC.
- Report the **primary vs. benchmark** (`INCLUDE_LC_RISK`) gap.
- Threshold tuning, calibration, lift/gains and expected-loss/ROI translation, and
  the policy-characterization model with/without the `Risk_Score_missing` flag.